# Activity

O kernel "poly" do SVC na verdade tem outro atributo para o número de graus do polinômio usado, que por padrão é 3. Por exemplo: `svm.SVC(kernel='poly', degree=3, C=1)`.

*(O print da atividade cortou aqui, mas seguindo a linha da aula, a ideia é testar diferentes valores de `degree` usando cross-validation, pra ver se o grau 3 padrão está mesmo dando overfit ou se dá pra achar um valor melhor.)*

In [1]:
# Mesma preparação da aula
import numpy as np
from sklearn import model_selection
from sklearn import datasets
from sklearn import svm

iris = datasets.load_iris()

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    iris.data, iris.target, test_size=0.4, random_state=0
)

In [2]:
# Relembrando os resultados da aula pra comparação:
# kernel linear -> 0.9667 no split único, 0.98 na média do cross-validation
# kernel poly (degree=3, padrão) -> 0.98 no cross-validation, mas só 0.9 no split único
# essa diferença é o sinal de que o poly com degree=3 pode estar dando overfit
clf = svm.SVC(kernel='poly', degree=3, C=1).fit(X_train, y_train)
print('poly degree=3 - score no split único:', clf.score(X_test, y_test))

scores = model_selection.cross_val_score(clf, iris.data, iris.target, cv=5)
print('poly degree=3 - scores do cross-validation:', scores)
print('poly degree=3 - média do cross-validation:', scores.mean())

poly degree=3 - score no split único: 0.9
poly degree=3 - scores do cross-validation: [0.96666667 1.         0.96666667 0.96666667 1.        ]
poly degree=3 - média do cross-validation: 0.9800000000000001


In [3]:
# Agora testando vários valores de degree, sempre comparando split único com cross-validation
# pra ver em qual grau o modelo deixa de dar overfit
degrees = [1, 2, 3, 4, 5, 6]

for degree in degrees:
    clf = svm.SVC(kernel='poly', degree=degree, C=1).fit(X_train, y_train)
    split_score = clf.score(X_test, y_test)

    cv_scores = model_selection.cross_val_score(clf, iris.data, iris.target, cv=5)
    cv_mean = cv_scores.mean()

    print(f'degree = {degree} -> split único: {split_score:.4f} | média cross-val: {cv_mean:.4f} | diferença: {abs(split_score - cv_mean):.4f}')

degree = 1 -> split único: 0.9000 | média cross-val: 0.9533 | diferença: 0.0533
degree = 2 -> split único: 0.9500 | média cross-val: 0.9867 | diferença: 0.0367
degree = 3 -> split único: 0.9000 | média cross-val: 0.9800 | diferença: 0.0800
degree = 4 -> split único: 0.9167 | média cross-val: 0.9667 | diferença: 0.0500
degree = 5 -> split único: 0.9500 | média cross-val: 0.9733 | diferença: 0.0233
degree = 6 -> split único: 0.9500 | média cross-val: 0.9533 | diferença: 0.0033


**Observação:** quanto maior a diferença entre o score do split único e a média do cross-validation, maior o sinal de overfitting (o modelo decorou o treino em vez de generalizar). O `degree` com a menor diferença entre os dois é o que generaliza melhor — vale comparar esse valor com o resultado do kernel linear (0.9667 / 0.98) pra ver se o poly consegue de fato superar ele sem dar overfit.